In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 超参数定义与设备选择
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
learning_rate = 0.001
epochs = 5

# 2. 数据准备 (MNIST 数据集下载与预处理)
transform = transforms.Compose([
    transforms.ToTensor(), # 将图片转化为 Tensor，且像素值归一化到 [0, 1]
    transforms.Normalize((0.1307,), (0.3081,)) # 标准化 (均值与标准差)
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# 3. 构建多层感知机 (MLP) 全连接网络
class FCNet(nn.Module):
    def __init__(self):
        super(FCNet, self).__init__()
        # 展平后 28x28 = 784 维特征
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        # 最终输出 10 个特征/Logits，对应数字 0-9
        """
        nn.Linear(64, 10)：输入64维特征，输出10个logits
        公式：
            z = W * x + b
            W：[10, 64]权重矩阵
            b：长度为10的偏置
            输出向量z的长度=10，每个位置对应一个类别得分
        """
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        # 将输入张量 [batch_size, 1, 28, 28] 展平为 [batch_size, 784]
        x = x.view(-1, 28 * 28)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # 输出 10 个特征
        return x

model = FCNet().to(device)

# 4. 定义损失函数与优化器
criterion = nn.CrossEntropyLoss()  # 内部自动集成了 Softmax 计算

# 优化器在初始化时，提前保存了所有参数张量的引用；在后面optimizer.step()时，读取张量上已经算好的.grad，修改张量的.data（权重数值）
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 5. 模型训练
for epoch in range(epochs):
    """
    model.train()：把模型切换到训练模式
        Dropout启用；BN层使用batch内均值方差，并且更新滑动统计量
        只是修改self.training=True，不会开始训练，不会计算任何loss
    """
    model.train()
    """
    running_loss = 0.0：
        running_loss只是累加损失的变量，初始化为0
        用途：记录一个epoch内所有batch损失的总和，最后用来求平均损失
    """
    running_loss = 0.0
    # 循环938次 len(train_loader) = 938 -> len(train_dataset) = 60000 / batch_size = 64
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        """
        同一个forward图里，一个参数被多次使用
            z = x * y + x ** 2                      x用了两次
            z = backward()                          x的梯度 = ∂z/∂x = y + 2x，是两路梯度的和
        反向传播到x时，有两个上游分支（x*y一路、x²一路）。引擎把这两路梯度累加到x.grad上，数学上这本来就是求和（链式法则的加法规则）。如果用“覆盖”，第二路会把第一路冲掉，结果就错了。所以 += 是反向传播数学结构的自然结果
        """
        optimizer.zero_grad()        # 梯度清零
        """
        核心原理：nn.Module重写了__call__魔法方法
        class Module:
            def __call__(self, *args, **kwargs):
                # 【这里会先执行一堆钩子（hook）】
                # 然后调用你写的 forward
                result = self.forward(*args, **kwargs)
                return result
        Python里，对象加括号obj(xxx)，本质是调用这个对象的__call__方法
        注意：
            这个outputs（即logits本身就是张量）
        """
        outputs = model(images)      # 前向传播得到 10 个特征
        """
        loss是一个PyTorch张量（Tensor），不是单纯一个数字
            内容：
                张量里包含损失的标量数值（整个batch的平均损失）
                同时附带计算图信息：记录这个损失是怎么从outputs、模型权重一步步算出来的（梯度计算需要的依赖关系）
            一句话：
                loss = 损失数值 + 计算图
        """
        loss = criterion(outputs, labels) # 计算交叉熵损失，logits（原始得分）
        
        # 使用loss里保存的计算图，反向求导，给模型参数算出梯度，存到param.grad
        """
        param.grad不是model的属性，是参数张量（Tensor）自己的属性，model只是持有这些张量的引用
            outputs本身也是张量
        loss.backward()顺着计算图找到所有参与计算的张量，直接修改张量内部的.grad字段
        
        前向产生的中间张量（激活值）：
            只要你还在构建计算图（没torch.no_grad()），前向所有中间结果必须保存下来，留给反向求导
            前向：计算 x1 = fc1(x0), x2 = fc2(x1), x3 = fc3(x2)...
            autograd会保留x1, x2这些中间激活张量（也就是中间的结果特征张量），存到内存，直到loss.backward()执行完毕
            backward跑完之后，这些中间张量（因为中间梯度已经被计算出来了，后面optimizer.step()可以直接根据梯度值更新梯度，不需要这些中间结果特征张量了）自动被销毁、内存释放，只剩下叶子参数（weight/bias）还留在内存
        
        例子：
            计算图是反向链式树，根节点是loss，叶子节点就是叶子张量（weight/bias）
            loss 【根节点，运算输出，非叶子张量】
            └── NllLossBackward0
                └── LogSoftmaxBackward
                    └── AddmmBackward0 (fc3线性层运算)
                        ├─ ReluBackward0 (fc2的relu输出，中间张量)
                        │   └── AddmmBackward0 (fc2线性运算)
                        │       ├─ AddmmBackward0 (fc1线性运算)
                        │       │   ├─ 输入images（requires_grad=False，不参与求导）
                        │       │   ├─ fc1.bias 【🍃叶子参数】
                        │       │   └─ fc1.weight 【🍃叶子参数】
                        │       ├─ fc2.bias 【🍃叶子参数】
                        │       └─ fc2.weight 【🍃叶子参数】
                        ├─ fc3.bias 【🍃叶子参数】
                        └─ fc3.weight 【🍃叶子参数】
        """
        loss.backward()              # 反向传播
        """
            优化器在初始化时，提前保存了所有参数张量的引用
            optimizer.step()时，读取张量上已经算好的.grad，修改张量的.data（权重数值）
        """
        optimizer.step()             # 更新参数
        """
        loss.item()：取出loss张量里面的Python数值，如果直接写 running_loss += loss 会构建计算图，占用大量内存！必须加.item()
            
        loss：是PyTorch张量（tensor），带有计算图、梯度信息
            
        .item()：从loss张量里面取出普通python浮点数（损失的纯数字），脱离计算图
            
        为什么放在epoch循环内：
            每一轮epoch都要清零running_loss，不然会不断累积前面所有epoch的损失，数值越来越大
        """
        running_loss += loss.item()
    
    # len(train_loader) = 一个epoch里面一共有多少个batch
    # running_loss / len(train_loader)：求一个epoch里的平均损失
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# 6. 模型测试 (验证识别准确率)
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        # images, labels 数据类型都属于 <class 'torch.Tensor'>
        """
            images.to(device)/ labels.to(device)
        作用：
            把张量数据从原来的设备（默认CPU）迁移到自定设备device（GPU/cpu）
        PyTorch要求：
            模型和输入数据、标签，必须在同一个设备上才能计算，否则直接报错
        """
        images, labels = images.to(device), labels.to(device)
        # logits（原始得分）
        outputs = model(images)
        
        # 通过 argmax 取 10 个输出特征中最大值的索引，即为预测数字
        """
        在深度学习训练和预测中，模型通常是批次（Batch）输入数据的，因此输出张量（outputs）是一个2维张量，其形状为[batch_size, 10]：
            维度0（dim=0）：代表Batch（样本）方向。沿着这个方向找最大值，会得到一个Batch里的所有图片在某个类别上的最大得分（通常没有实际预测意义）
            维度1（dim=1）：代表Classes（类别）方向，也就是对应数字0到9的10个得分特征
        因此，传入1就是告诉PyTorch：对每张图片（按行遍历），在它的10个类别得分中找到最大值
        """
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"测试集准确率: {100 * correct / total:.2f}%")

Epoch [1/5], Loss: 0.4785
Epoch [2/5], Loss: 0.3488
Epoch [3/5], Loss: 0.3136
Epoch [4/5], Loss: 0.2900
Epoch [5/5], Loss: 0.2728
测试集准确率: 87.70%


In [14]:
# 这里是test_loader
print(type(images))
print(len(images))

print(type(labels))
print(len(labels))

print(type(outputs))
print(len(outputs))

<class 'torch.Tensor'>
16
<class 'torch.Tensor'>
16
<class 'torch.Tensor'>
16


In [9]:
"""
直观对比示例：
    假设Batch大小为2（有两张图片），模型输出outputs如下：
"""
# outputs 的形状是 [2, 10]  [batch_size, num_classes]
outputs = torch.tensor([
    [0.1, 0.2, 0.0, 0.85, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # 第 1 张图片，索引 3 数值最大
    [0.0, 0.9, 0.1, 0.00, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]   # 第 2 张图片，索引 1 数值最大
])
"""
values：每一行最大数值/最高得分是多少
predicted：最大数值所在索引位置（index），即预测的数字是几（0～9）
torch.max()与torch.argmax()的区别：
    相同点：
        都是Pytorch中寻找张量（Tensor）中最大值的函数，两者的核心区别在于返回值内容不同
    核心区别：
        torch.max：既返回最大值本身（values），也返回最大值所在的索引（indices）
        torch.argmax：只返回最大值所在的索引（indices）
"""
values, predicted = torch.max(outputs, 1)

print(values)     # 输出: tensor([0.85, 0.90]) -> 对应的最大得分
print(predicted)  # 输出: tensor([3, 1])      -> 沿着 dim=1 找到的最大值索引（即预测结果）

tensor([0.8500, 0.9000])
tensor([3, 1])


In [11]:
print(len(train_dataset))
print(type(train_dataset))

print(len(train_loader))
print(type(train_loader))

60000
<class 'torchvision.datasets.mnist.FashionMNIST'>
938
<class 'torch.utils.data.dataloader.DataLoader'>


In [7]:
60000 / 64

937.5

In [ ]:
"""
pred = model(x)            # ① 前向：所有层用"当前"权重算一遍
loss = criterion(pred, y)  #    （此时所有权重一动不动）

loss.backward()            # ② 反向：算出"每一层权重"的梯度（还是不更新！）      loss.backward()期间存梯度

optimizer.step()           # ③ 更新：所有层权重同时按梯度改一步                 optimizer.step()期间更新权重

梯度计算的的顺序是从最后一层往第一层倒着走，每一站的权重梯度当场算出，存进该权重（叶子）的 .grad
    # h = ReLU(x @ W1 + b1)     第 1 层权重：W1（叶子）
    # out = h @ W2 + b2         第 2 层权重：W2（叶子）
    # loss = MSE(out, target)
loss.backward()内部发生的事：
    起点：loss（上游梯度 = 1）

    第 1 站：MSE 的反向节点
        算出 ∂loss/∂out，传给 out
    
    第 2 站：out 的 AddmmBackward（h @ W2 + b2）
        ├─ 算出 ∂loss/∂W2 = (∂loss/∂out) · hᵀ  → 写入 W2.grad ✓（第 2 层权重的梯度到手！）
        ├─ 算出 ∂loss/∂b2                        → 写入 b2.grad ✓
        └─ 算出 ∂loss/∂h = (∂loss/∂out) · W2ᵀ，继续往回传
                              ↑ 注意：这里用的是 W2 的旧值，
                                整个反向过程中 W2 都没变过
    
    第 3 站：ReLU 的反向节点
        算出 ∂loss/∂(x@W1+b1)，继续往回传
    
    第 4 站：x @ W1 + b1 的反向节点
        ├─ 算出 ∂loss/∂W1 = (∂loss/∂z1) · xᵀ   → 写入 W1.grad ✓（第 1 层权重的梯度到手！）
        └─ 算出 ∂loss/∂b1                        → 写入 b1.grad ✓
    
    终点：x 是数据，不需要梯度
    
两个关键点：
    每一层的权重梯度，在自己的那一站就被存进 .grad 了——所以“中间层的权重”和输出层权重一样，梯度都完好无损，这就是所说的“梯度在终点站落地”
    整个backward期间，所有权重都是旧值。第4站算W1的梯度时用的W2，和第一站时用的W2是同一个值——因为还没有人改它
"""

In [1]:
"""
x = torch.tensor([2.0])
w = torch.tensor([3.0], requires_grad=True)   # w 是叶子（参数）

y = x * w      # 中间张量（激活值），非叶子
loss = y ** 2  # loss = (xw)² = 36
loss.backward()

backward 完整计算链：
    第1站：loss 的 PowBackward0
        收到上游梯度 1
        算出：∂loss/∂y = 2y = 12      ← y 的梯度在这里被算出来了！
        传给 y

    第2站：y 的 MulBackward0
            收到 ∂loss/∂y = 12
            算出：∂loss/∂w = 12 · x = 24
            传给 w
    
    第3站：w 的 AccumulateGrad
            w.grad += 24                    ← 只有这里"存"了下来
            
y的梯度（12）在第一站就被算出来了，它是链式法则不可或缺的中间结果 —— 没有它就算不出w的梯度。但这个12算完后立刻被第2站消费掉，没有写进任何 .grad，因为没人需要
中间层的激活值：作为反向传播的“中转站”，每站都要算出自己梯度（否则链式法则断链）
中间层的权重（nn.linear的weight、bias）：他们是叶子，梯度会正常存进 .grad ——每一层参数梯度都完好无损
"""

NameError: name 'torch' is not defined